# 12. Mapping Review Workflow — Approve, Reject, Override

---

This notebook demonstrates the **local mapping review workflow** — how to review, approve, reject, and override individual schema and concept mappings from Python code.

**What you'll learn:**
- Review schema mappings: `summary()`, `needs_review()`, `auto_accepted()`
- Approve, reject, or override individual mappings
- Export mappings to CSV for clinical SME review
- Reload edited CSV back into the SDK
- Save reviewed mappings back to project storage
- Same workflow for concept mappings

**Prerequisites:**
- `pip install portiere-health[faiss] pandas`

## 1. Setup & Run Initial Mapping

In [1]:
from pathlib import Path

ASSETS_DIR = Path("example_assets/12_mapping_review_workflow")
PATIENTS_CSV = ASSETS_DIR / "patients.csv"
DIAGNOSES_CSV = ASSETS_DIR / "diagnoses.csv"

for f in [PATIENTS_CSV, DIAGNOSES_CSV]:
    assert f.exists(), f"Missing: {f}"
print("All sample data files found.")

All sample data files found.


In [2]:
from portiere.knowledge import build_knowledge_layer

ATHENA_DIR = Path("example_assets/vocabulary_download_v5")
VOCAB_DIR  = ATHENA_DIR / "indexes"
BM25S_CORPUS = VOCAB_DIR / "concepts.json"
FAISS_INDEX  = VOCAB_DIR / "concepts.index"
FAISS_META   = VOCAB_DIR / "concepts.meta.json"

# Build indexes from Athena on first run (~10-30 min for FAISS embedding)
if not BM25S_CORPUS.exists():
    paths = build_knowledge_layer(
        athena_path=ATHENA_DIR,
        output_path=VOCAB_DIR,
        backend="hybrid",
        vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    )
    print(f"Built indexes: {paths}")
else:
    print(f"Using existing indexes at {VOCAB_DIR}")

Using existing indexes at example_assets/vocabulary_download_v5/indexes


In [3]:
import portiere
from portiere.config import (
    PortiereConfig,
    KnowledgeLayerConfig,
    EmbeddingConfig,
    RerankerConfig,
)
from portiere.engines import PolarsEngine

config = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="faiss",
        faiss_index_path=FAISS_INDEX,
        faiss_metadata_path=FAISS_META,
    ),
    embedding=EmbeddingConfig(
        provider="huggingface",
        model="cambridgeltl/SapBERT-from-PubMedBERT-fulltext",
    ),
    reranker=RerankerConfig(provider="none"),
)

project = portiere.init(
    name="Review Workflow Demo",
    engine=PolarsEngine(),
    target_model="omop_cdm_v5.4",
    config=config,
)
print(project)

2026-04-16 00:27:06 [info     ] PolarsEngine initialized      


2026-04-16 00:27:06 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


Project(name='Review Workflow Demo', task='standardize', target_model='omop_cdm_v5.4', mode='local')


In [4]:
# Add sources and run schema mapping
patients = project.add_source(str(PATIENTS_CSV))
diagnoses = project.add_source(str(DIAGNOSES_CSV))

schema_map = project.map_schema(patients)
print("Initial schema mapping:")
print(schema_map.summary())

2026-04-16 00:27:06 [info     ] project.source_added           project='Review Workflow Demo' source=patients


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-16 00:27:08 [info     ] gx_profiler.profiled           columns=11 rows=15 source=patients


2026-04-16 00:27:08 [info     ] project.profiled               source=patients


2026-04-16 00:27:08 [info     ] project.source_added           project='Review Workflow Demo' source=diagnoses


...

2026-04-16 00:27:10 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:27:10 [info     ] Stage 2 complete               auto=10 review=0 total=11


2026-04-16 00:27:10 [info     ] local_storage.schema_mapping_saved items_count=11 project='Review Workflow Demo'


2026-04-16 00:27:10 [info     ] project.schema_mapped          auto=10 project='Review Workflow Demo' total=11


Initial schema mapping:
{'total': 11, 'auto_accepted': 10, 'needs_review': 0, 'approved': 0, 'unmapped': 1, 'auto_rate': 90.9090909090909}


## 2. Review Schema Mappings

After `map_schema()`, each mapping item has a status based on confidence:
- **auto_accepted** (>= 0.95): High confidence, accepted automatically
- **needs_review** (0.70 - 0.95): Medium confidence, needs human review
- **unmapped** (< 0.70): Low confidence, needs manual mapping

In [5]:
# Summary of mapping statuses
summary = schema_map.summary()
print(f"Total mappings:    {summary['total']}")
print(f"Auto-accepted:     {summary['auto_accepted']}")
print(f"Needs review:      {summary['needs_review']}")
print(f"Unmapped:          {summary['unmapped']}")
print(f"Auto-accept rate:  {summary['auto_rate']:.1f}%")

Total mappings:    11
Auto-accepted:     10
Needs review:      0
Unmapped:          1
Auto-accept rate:  90.9%


In [6]:
# Browse auto-accepted items
print("Auto-accepted mappings:")
print(f"{'Source Column':25s} {'Target Table':25s} {'Target Column':30s} {'Confidence':>10s}")
print("-" * 95)
for item in schema_map.auto_accepted():
    print(f"{item.source_column:25s} {item.target_table or '':25s} {item.target_column or '':30s} {item.confidence:10.2f}")

Auto-accepted mappings:
Source Column             Target Table              Target Column                  Confidence
-----------------------------------------------------------------------------------------------
patient_id                person                    person_id                            0.95
first_name                person                    person_source_value                  0.95
date_of_birth             person                    birth_datetime                       0.95
gender                    person                    gender_concept_id                    0.95
race                      person                    race_concept_id                      0.95
ethnicity                 person                    ethnicity_concept_id                 0.95
address                   location                  address_1                            0.95
city                      location                  city                                 0.95
state                     location

In [7]:
# Browse items that need review
print("Items needing review:")
print(f"{'Source Column':25s} {'Target Table':25s} {'Target Column':30s} {'Confidence':>10s}")
print("-" * 95)
for item in schema_map.needs_review():
    print(f"{item.source_column:25s} {item.target_table or '':25s} {item.target_column or '':30s} {item.confidence:10.2f}")
    if item.candidates:
        print(f"  Candidates:")
        for c in item.candidates[:3]:
            print(f"    → {c.get('target_table', '')}.{c.get('target_column', '')} (score={c.get('confidence', 0):.2f})")

Items needing review:
Source Column             Target Table              Target Column                  Confidence
-----------------------------------------------------------------------------------------------


## 3. Approve, Reject, Override Schema Mappings

Review individual items and set their status:

In [8]:
# Approve a mapping — accept the AI suggestion
# (Use the source_column name to find the item)
try:
    schema_map.approve("patient_id")
    item = schema_map.get_item("patient_id")
    print(f"Approved: patient_id → {item.effective_target_table}.{item.effective_target_column}")
    print(f"  Status: {item.status.value}")
except KeyError as e:
    print(f"Item not found: {e}")

Approved: patient_id → person.person_id
  Status: approved


In [9]:
# Override a mapping — choose a different target
try:
    schema_map.override(
        "date_of_birth",
        target_table="person",
        target_column="birth_datetime",
    )
    item = schema_map.get_item("date_of_birth")
    print(f"Overridden: date_of_birth → {item.effective_target_table}.{item.effective_target_column}")
    print(f"  Status: {item.status.value}")
    print(f"  Original suggestion: {item.target_table}.{item.target_column}")
    print(f"  Override: {item.override_target_table}.{item.override_target_column}")
except KeyError as e:
    print(f"Item not found: {e}")

Overridden: date_of_birth → person.birth_datetime
  Status: overridden
  Original suggestion: person.birth_datetime
  Override: person.birth_datetime


In [10]:
# Reject a mapping — mark as not mappable
try:
    schema_map.reject("zip_code")
    item = schema_map.get_item("zip_code")
    print(f"Rejected: zip_code")
    print(f"  Status: {item.status.value}")
except KeyError as e:
    print(f"Item not found: {e}")

Rejected: zip_code
  Status: rejected


In [11]:
# Check updated summary
print("Updated summary after review:")
summary = schema_map.summary()
for key, value in summary.items():
    print(f"  {key}: {value}")

Updated summary after review:
  total: 11
  auto_accepted: 7
  needs_review: 0
  approved: 1
  unmapped: 1
  auto_rate: 63.63636363636363


## 4. Export to CSV for SME Review

Export mappings to CSV so clinical subject matter experts can review them in Excel or Google Sheets.

### Example: Schema Mapping CSV

The exported CSV contains one row per source column with the AI's suggested mapping and status. SMEs can edit the `status`, `override_target_table`, and `override_target_column` columns directly:

| source_column | source_table | target_table | target_column | confidence | status | override_target_table | override_target_column |
|:---|:---|:---|:---|---:|:---|:---|:---|
| patient_id | | person | person_id | 0.95 | approved | | |
| first_name | | person | person_source_value | 0.95 | needs_review | | |
| last_name | | person | person_source_value | 0.50 | needs_review | | |
| date_of_birth | | person | birth_datetime | 0.95 | approved | person | birth_datetime |
| gender | | person | gender_concept_id | 0.95 | needs_review | | |
| race | | person | race_concept_id | 0.95 | needs_review | | |
| ethnicity | | person | ethnicity_concept_id | 0.95 | needs_review | | |
| address | | location | address_1 | 0.95 | needs_review | | |
| city | | location | city | 0.95 | needs_review | | |
| state | | location | state | 0.95 | needs_review | | |
| zip_code | | location | zip | 0.95 | rejected | | |

### Example: Concept Mapping CSV

The concept mapping CSV contains one row per source code. SMEs can approve, reject, or override each mapping:

| source_code | source_description | source_vocabulary | target_concept_id | target_concept_name | target_vocabulary_id | confidence | method | status |
|:---|:---|:---|---:|:---|:---|---:|:---|:---|
| E11.9 | Type 2 diabetes mellitus without complications | ICD10CM | 201826 | Type 2 diabetes mellitus | SNOMED | 0.99 | AUTO | auto_mapped |
| I10 | Essential (primary) hypertension | ICD10CM | 320128 | Essential hypertension | SNOMED | 0.97 | AUTO | auto_mapped |
| J18.9 | Pneumonia unspecified organism | ICD10CM | 255848 | Pneumonia | SNOMED | 0.92 | VERIFY | needs_review |
| N18.6 | End stage renal disease | ICD10CM | 193782 | End-stage renal disease | SNOMED | 0.88 | VERIFY | needs_review |
| Z87.891 | Personal history of nicotine dependence | ICD10CM | 37109023 | Nicotine dependence | SNOMED | 0.72 | REVIEW | needs_review |

**SME workflow:**
1. Open the CSV in Excel or Google Sheets
2. Review each row — check the AI's suggested `target_concept_name`
3. Change `status` to `approved`, `rejected`, or leave as `needs_review`
4. For overrides: fill in `override_target_table` / `override_target_column` (schema) or the correct `target_concept_id` (concepts)
5. Save and reload into the SDK with `SchemaMapping.from_csv()` or `ConceptMapping.from_csv()`

In [12]:
# Export to DataFrame
df = schema_map.to_dataframe()
print("Schema mapping as DataFrame:")
df

Schema mapping as DataFrame:


source_column,source_table,target_table,target_column,confidence,status
str,str,str,str,f64,str
"""patient_id""","""""","""person""","""person_id""",0.95,"""approved"""
"""first_name""","""""","""person""","""person_source_value""",0.95,"""auto_accepted"""
"""last_name""","""""","""person""","""person_source_value""",0.5,"""unmapped"""
"""date_of_birth""","""""","""person""","""birth_datetime""",0.95,"""overridden"""
"""gender""","""""","""person""","""gender_concept_id""",0.95,"""auto_accepted"""
…,…,…,…,…,…
"""ethnicity""","""""","""person""","""ethnicity_concept_id""",0.95,"""auto_accepted"""
"""address""","""""","""location""","""address_1""",0.95,"""auto_accepted"""
"""city""","""""","""location""","""city""",0.95,"""auto_accepted"""


In [13]:
# Export to CSV file
output_csv = "schema_review.csv"
schema_map.to_csv(output_csv)
print(f"Exported to {output_csv}")

# Show the CSV content
import pandas as pd
print()
print(pd.read_csv(output_csv).to_string())

Exported to schema_review.csv

    source_column  source_table target_table         target_column  confidence         status
0      patient_id           NaN       person             person_id        0.95       approved
1      first_name           NaN       person   person_source_value        0.95  auto_accepted
2       last_name           NaN       person   person_source_value        0.50       unmapped
3   date_of_birth           NaN       person        birth_datetime        0.95     overridden
4          gender           NaN       person     gender_concept_id        0.95  auto_accepted
5            race           NaN       person       race_concept_id        0.95  auto_accepted
6       ethnicity           NaN       person  ethnicity_concept_id        0.95  auto_accepted
7         address           NaN     location             address_1        0.95  auto_accepted
8            city           NaN     location                  city        0.95  auto_accepted
9           state           N

## 5. Reload from Edited CSV

After the SME edits the CSV (e.g., changes statuses, corrects target columns), reload it back into the SDK.

In [14]:
from portiere.models.schema_mapping import SchemaMapping

# Simulate SME editing the CSV:
# (In real use, the SME would edit in Excel and save)
df = pd.read_csv(output_csv)

# SME approves gender mapping
df.loc[df["source_column"] == "gender", "status"] = "approved"

# Save edited CSV
edited_csv = "schema_review_edited.csv"
df.to_csv(edited_csv, index=False)
print(f"SME edited and saved: {edited_csv}")

# Reload from edited CSV
reviewed_schema = SchemaMapping.from_csv(edited_csv)
print(f"\nReloaded {len(reviewed_schema.items)} mappings")
print(reviewed_schema.summary())

SME edited and saved: schema_review_edited.csv

Reloaded 11 mappings
{'total': 11, 'auto_accepted': 6, 'needs_review': 0, 'approved': 2, 'unmapped': 1, 'auto_rate': 54.54545454545454}


## 6. Save Reviewed Mappings

Save the reviewed mapping back to the project's storage (local YAML files).

In [15]:
# Save reviewed schema mapping to project storage
project.save_schema_mapping(reviewed_schema)
print("Saved reviewed schema mapping to project storage.")

# Verify by reloading
reloaded = project.load_schema_mapping()
print(f"\nReloaded from storage: {len(reloaded.items)} items")
print(reloaded.summary())

2026-04-16 00:27:10 [info     ] local_storage.schema_mapping_saved items_count=11 project='Review Workflow Demo'


2026-04-16 00:27:10 [info     ] project.schema_mapping_saved   items=11 project='Review Workflow Demo'


Saved reviewed schema mapping to project storage.

Reloaded from storage: 11 items
{'total': 11, 'auto_accepted': 6, 'needs_review': 0, 'approved': 2, 'unmapped': 1, 'auto_rate': 54.54545454545454}


---

## 7. Concept Mapping Review

The same review workflow works for concept mappings.

In [16]:
# Run concept mapping
concept_map = project.map_concepts(source=diagnoses)
print("Concept mapping summary:")
print(concept_map.summary())

2026-04-16 00:27:11 [info     ] project.code_columns_detected  columns=['diagnosis_code']


2026-04-16 00:27:11 [info     ] Stage 3: Concept mapping (local) columns=['diagnosis_code'] knowledge_backend=faiss


2026-04-16 00:27:11 [info     ] Mapping column: diagnosis_code


2026-04-16 00:27:11 [info     ] Found description column: diagnosis_description code_column=diagnosis_code desc_column=diagnosis_description mappings=8


2026-04-16 00:27:11 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:27:11 [info     ] knowledge.creating_backend     backend=faiss model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext


/Users/kuangsmacbook/Downloads/CUSP/PORTIERE/portiere/src/portiere/engines/polars_engine.py:131: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count()
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


2026-04-16 00:27:12 [info     ] faiss.index_loaded             concepts_count=623910 index_size=623910


2026-04-16 00:27:12 [info     ] local_concept_mapper.initialized code_index_entries=0 knowledge_backend=LocalFAISSBackend llm_verifier=False reranker=False


ImportError: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers

In [17]:
# Browse auto-mapped concepts
print("Auto-mapped concepts:")
print(f"{'Source Code':12s} {'Source Description':45s} {'Target Concept':35s} {'Confidence':>10s}")
print("-" * 110)
for item in concept_map.auto_mapped():
    print(f"{item.source_code:12s} {(item.source_description or '')[:45]:45s} "
          f"{(item.target_concept_name or '')[:35]:35s} {item.confidence:10.2f}")

Auto-mapped concepts:
Source Code  Source Description                            Target Concept                      Confidence
--------------------------------------------------------------------------------------------------------------


NameError: name 'concept_map' is not defined

In [18]:
# Browse items needing review
print("Concepts needing review:")
print(f"{'Source Code':12s} {'Description':40s} {'Top Candidate':35s} {'Confidence':>10s}")
print("-" * 100)
for item in concept_map.needs_review():
    print(f"{item.source_code:12s} {(item.source_description or '')[:40]:40s} "
          f"{(item.target_concept_name or '')[:35]:35s} {item.confidence:10.2f}")

Concepts needing review:
Source Code  Description                              Top Candidate                       Confidence
----------------------------------------------------------------------------------------------------


NameError: name 'concept_map' is not defined

In [19]:
# Approve a concept mapping
try:
    concept_map.approve("E11.9")
    item = concept_map.get_item("E11.9")
    print(f"Approved: E11.9 → {item.target_concept_name} (concept_id={item.target_concept_id})")
    print(f"  Method: {item.method.value}")
except KeyError as e:
    print(f"Code not found: {e}")

NameError: name 'concept_map' is not defined

In [20]:
# Override a concept mapping with a specific concept ID
try:
    concept_map.override(
        "I10",
        concept_id=320128,
        concept_name="Essential hypertension",
        vocabulary_id="SNOMED",
    )
    item = concept_map.get_item("I10")
    print(f"Overridden: I10 → {item.target_concept_name} (concept_id={item.target_concept_id})")
    print(f"  Method: {item.method.value}")
except KeyError as e:
    print(f"Code not found: {e}")

NameError: name 'concept_map' is not defined

In [21]:
# Reject a concept mapping
try:
    concept_map.reject("Z87.891")
    item = concept_map.get_item("Z87.891")
    print(f"Rejected: Z87.891 — {item.source_description}")
    print(f"  Method: {item.method.value}")
except KeyError as e:
    print(f"Code not found: {e}")

NameError: name 'concept_map' is not defined

In [22]:
# Updated summary
print("Updated concept mapping summary:")
print(concept_map.summary())

Updated concept mapping summary:


NameError: name 'concept_map' is not defined

## 8. Export & Import Concept Mappings

In [23]:
# Export to DataFrame
concept_df = concept_map.to_dataframe()
print("Concept mapping as DataFrame:")
concept_df

NameError: name 'concept_map' is not defined

In [24]:
# Export to CSV
concept_map.to_csv("concept_review.csv")
print("Exported to concept_review.csv")

# Show OMOP source_to_concept_map format
stcm = concept_map.to_source_to_concept_map()
print(f"\nOMOP source_to_concept_map: {len(stcm)} rows")
if stcm:
    print(pd.DataFrame(stcm).to_string())

NameError: name 'concept_map' is not defined

In [25]:
# Reload from CSV
from portiere.models.concept_mapping import ConceptMapping

reloaded_concepts = ConceptMapping.from_csv("concept_review.csv")
print(f"Reloaded {len(reloaded_concepts.items)} concept mappings from CSV")
print(reloaded_concepts.summary())

FileNotFoundError: [Errno 2] No such file or directory: 'concept_review.csv'

In [26]:
# Save reviewed concept mappings to project
project.save_concept_mapping(concept_map)
print("Saved reviewed concept mapping to project storage.")

NameError: name 'concept_map' is not defined

## 9. Bulk Review Operations

In [27]:
# Approve all remaining items that need review
remaining = len(concept_map.needs_review())
concept_map.approve_all()
print(f"Bulk approved {remaining} items")
print(concept_map.summary())

NameError: name 'concept_map' is not defined

In [28]:
# Finalize mapping (locks it for ETL generation)
concept_map.finalize()
print(f"Finalized: {concept_map.finalized}")

NameError: name 'concept_map' is not defined

---

## Review Workflow Cheat Sheet

### Schema Mapping

```python
# Browse
schema_map.summary()             # Status counts
schema_map.needs_review()        # Items needing review
schema_map.auto_accepted()       # Auto-accepted items
schema_map.rejected()            # Rejected items
schema_map.overridden()          # Overridden items
schema_map.get_item("col")       # Get specific item

# Review actions
schema_map.approve("col")        # Accept AI suggestion
schema_map.reject("col")         # Mark as not mappable
schema_map.override("col", target_table="...", target_column="...")
schema_map.approve_all()         # Bulk approve all needs_review

# Export / Import
schema_map.to_dataframe()        # pandas DataFrame
schema_map.to_csv("review.csv")  # Export for SME review
SchemaMapping.from_csv("edited.csv")  # Reload after SME edits

# Save
project.save_schema_mapping(schema_map)
```

### Concept Mapping

```python
# Browse
concept_map.summary()            # Status counts
concept_map.needs_review()       # Items needing review
concept_map.auto_mapped()        # Auto-mapped items
concept_map.unmapped()           # Unmapped items
concept_map.get_item("E11.9")    # Get specific item

# Review actions
concept_map.approve("E11.9")     # Accept AI suggestion
concept_map.reject("Z87.891")    # Mark as unmapped
concept_map.override("I10", concept_id=320128, concept_name="Essential hypertension")
concept_map.approve_all()        # Bulk approve all needs_review

# Export / Import
concept_map.to_dataframe()       # pandas DataFrame
concept_map.to_csv("review.csv") # Export for SME review
ConceptMapping.from_csv("edited.csv")  # Reload after SME edits
concept_map.to_source_to_concept_map() # OMOP format

# Save
project.save_concept_mapping(concept_map)
```

---

**Next steps:**
- See `11_hybrid_mode.ipynb` to push reviewed mappings to cloud
- See `08_end_to_end_omop.ipynb` for full ETL after review